In [2]:
import numpy as np
import os
import glob
from tqdm import tqdm

# --- Configuration ---
# 1. Set the directory where your individual .npz embedding files are located.
#    This should be the directory containing files like 'specialized_embeddings1.npz', etc.
INPUT_DIR = "/kaggle/input/test-fast-embeding"

# 2. Set the name and path for the final, combined output file.
OUTPUT_PATH = "/kaggle/working/all_embeddings_aggregated.npz"
# --- End Configuration ---

def aggregate_embeddings(input_directory, output_path):
    """
    Finds all .npz files in a directory, aggregates their contents,
    and saves them into a single compressed .npz file.
    """
    # Find all .npz files in the specified directory
    search_pattern = os.path.join(input_directory, "*.npz")
    file_paths = glob.glob(search_pattern)

    # In case this script is in the same directory, exclude the output file from the input list
    output_filename = os.path.basename(output_path)
    file_paths = [p for p in file_paths if os.path.basename(p) != output_filename]

    if not file_paths:
        print(f"Error: No .npz files found to aggregate in '{input_directory}'.")
        print("Please check that your INPUT_DIR path is correct.")
        return

    print(f"Found {len(file_paths)} embedding files to aggregate.")
    print("Files to be merged:", [os.path.basename(p) for p in file_paths])

    # Initialize lists to store the arrays from each file
    all_ids = []
    all_text_embeddings = []
    all_image_embeddings = []
    all_prices = []
    price_found_in_any_file = False  # Flag to check if at least one file has a price array

    # Loop through each file and load its contents
    for path in tqdm(file_paths, desc="Aggregating files"):
        try:
            data = np.load(path)
            # Always append the mandatory arrays
            all_ids.append(data['ids'])
            all_text_embeddings.append(data['text'])
            all_image_embeddings.append(data['image'])

            # Conditionally load the 'price' array only if it exists in the file
            if 'price' in data:
                all_prices.append(data['price'])
                price_found_in_any_file = True

        except Exception as e:
            print(f"\nWarning: Could not process file {os.path.basename(path)}. Error: {e}")
            continue

    # --- Concatenate all the collected arrays ---
    print("\nConcatenating arrays...")
    final_ids = np.concatenate(all_ids)
    final_text = np.concatenate(all_text_embeddings)
    final_image = np.concatenate(all_image_embeddings)

    # --- Prepare the dictionary of data to save ---
    data_to_save = {
        'ids': final_ids,
        'text': final_text,
        'image': final_image
    }

    # Only concatenate and add the price array if it was found in any of the source files
    if price_found_in_any_file:
        print("Price data found, adding to the final aggregated file.")
        if len(all_prices) != len(file_paths):
             print(f"Warning: Price data was only found in {len(all_prices)} out of {len(file_paths)} files.")
        final_price = np.concatenate(all_prices)
        data_to_save['price'] = final_price
    else:
        print("No price data found in any of the files.")

    # --- Save the final aggregated file ---
    print(f"Saving aggregated data to {OUTPUT_PATH}...")
    np.savez_compressed(output_path, **data_to_save)

    # --- Print final summary ---
    print("\n--- Aggregation Complete ---")
    print(f"Total records processed: {len(final_ids)}")
    print(f"Final Text Embeddings Shape: {final_text.shape}")
    print(f"Final Image Embeddings Shape: {final_image.shape}")
    if price_found_in_any_file:
        print(f"Final Price Array Shape: {final_price.shape}")
    print(f"✅ Aggregated file saved successfully to: {OUTPUT_PATH}")

if __name__ == '__main__':
    aggregate_embeddings(INPUT_DIR, OUTPUT_PATH)


Found 9 embedding files to aggregate.
Files to be merged: ['specialized_test_embeddings7.npz', 'specialized_test_embeddings6.npz', 'specialized_test_embeddings1.npz', 'specialized_test_embeddings5.npz', 'specialized_test_embeddings3.npz', 'specialized_test_embeddings8.npz', 'specialized_test_embeddings2.npz', 'specialized_test_embeddings0.npz', 'specialized_test_embeddings4.npz']


Aggregating files: 100%|██████████| 9/9 [00:04<00:00,  2.08it/s]



Concatenating arrays...
No price data found in any of the files.
Saving aggregated data to /kaggle/working/all_embeddings_aggregated.npz...

--- Aggregation Complete ---
Total records processed: 75000
Final Text Embeddings Shape: (75000, 384)
Final Image Embeddings Shape: (75000, 768)
✅ Aggregated file saved successfully to: /kaggle/working/all_embeddings_aggregated.npz
